In [ ]:
# Example: 4000 samples at one pixel
np.random.seed(42)
rmr_samples_pixel = np.random.normal(loc=32, scale=8, size=4000)

threshold = 21  # Class V boundary
n_below = np.sum(rmr_samples_pixel < threshold)
p_class_v = n_below / len(rmr_samples_pixel)

print(f"=== EXCEEDANCE PROBABILITY AT ONE PIXEL ===")
print(f"Samples: {len(rmr_samples_pixel)}")
print(f"Mean RMR: {rmr_samples_pixel.mean():.1f}")
print(f"Std RMR:  {rmr_samples_pixel.std():.1f}")
print(f"Samples with RMR < {threshold}: {n_below}")
print(f"P(Class V) = {p_class_v:.3f} = {p_class_v*100:.1f}%")
print(f"Standard error: {np.sqrt(p_class_v * (1-p_class_v) / len(rmr_samples_pixel)):.3f}")
print()
#→ With mean RMR = 32 and σ = 8, there's a ~8% chance of Class V.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# === LOAD DAY 4 RESULTS ===
# f_2d has shape (4000, 2500) — 4000 posterior samples at 2500 grid cells
# (Reuse from Day 4, or reload from saved trace)

# RMR class thresholds
thresholds = {
    'Class V (Very Poor)': 21,
    'Class IV (Poor)': 41,
    'Class III (Fair)': 61,
    'Class II (Good)': 81,
}

# Compute P(RMR < threshold) at EVERY grid cell
prob_maps = {}
for name, threshold in thresholds.items():
    # For each grid cell: what fraction of 4000 samples are below threshold?
    prob = (f_2d < threshold).mean(axis=0)  # Shape: (2500,)
    prob_maps[name] = prob.reshape(ny, nx)

    print(f"{name} (RMR < {threshold}):")
    print(f"  Min P: {prob.min():.3f}  Max P: {prob.max():.3f}  Mean P: {prob.mean():.3f}")

# === PLOT: Exceedance Probability Maps ===
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, (name, pmap) in zip(axes.flat, prob_maps.items()):
    threshold = thresholds[name]
    im = ax.contourf(xx, yy, pmap, levels=np.linspace(0, 1, 21),
                      cmap='RdYlGn_r')  # Reversed: red = high probability
    ax.scatter(x_coords, y_coords, c='black', s=50, marker='^', zorder=5)
    ax.set_title(f'P(RMR < {threshold}): {name}', fontsize=11, fontweight='bold')
    ax.set_aspect('equal')
    ax.set_xlim(0, 4)
    ax.set_ylim(0, 4)
    ax.grid(True, alpha=0.3)
    plt.colorbar(im, ax=ax, label='Probability')

plt.suptitle('Exceedance Probability Maps from GP Posterior', fontsize=14)
plt.tight_layout()
plt.savefig("exceedance_probability_maps.png", dpi=150)
plt.show()